# Lesson 5, Part 3

Geometric series *A* + *Ax*<sup>2</sup> + *Ax*<sup>3</sup> + ... converge to *A*/(1 - *x*) for |*x*|<1, 
although many rounds may be needed to trim the gap.

In [ ]:
import numpy as np

def geom_convergence(x,tol=1E-14):
    predicted = 1/(1-x)
    result = term = 1
    for n in range(1,1000):
        if abs(result - predicted) < tol:
            break
        term *= x
        result += term
    return n

for x in np.linspace(-0.9,0.9,19):
    print('x =',round(x,2),':',geom_convergence(x),'iterations to convergence')

The series 1 + 1/2 + 1/3 + 1/4 +... is unbounded but it diverges at a very slow pace.

In [ ]:
def sum_reciprocal(threshold):
    result = n = 0
    while result < threshold:
        n += 1
        result += 1/n
    return n

for threshold in np.linspace(4,16,4):
    print('Sum exceeds',threshold,'after',sum_reciprocal(threshold),'terms')

The binomial theorem makes repeated use of combin(*n,k*), 
which tells us how many distinct combinations of *k* objects can be formed out of *n* objects. 
This can be computed in various ways, including the built-in function **math.comb**.

In [ ]:
import math
from time import time as T

combin = lambda n,k: math.factorial(n)/math.factorial(k)/math.factorial(n-k)

def combin1(n,k):
    result = n
    for i in range(1,k):
        result *= (n - i) / (i + 1) 
    return result

iters = 10000
for n,k in (12,8),(20,8),(90,30):
    print('n=',n,', k=',k,', comb(n,k) =\n',math.comb(n,k),combin(n,k),combin1(n,k))
    tic = T()
    for iter in range(iters): math.comb(n,k)
    print('math.comb takes time',T()-tic); tic = T()
    for iter in range(iters): combin(n,k)
    print('combin takes time',T()-tic); tic = T()
    for iter in range(iters): combin1(n,k)
    print('combin1 takes time',T()-tic,'\n')

Remarkably, the ratio of products in combin(*n,k*) can be computed 
through repeated additions within a pyramid of numbers, 
where each element sums the two closest neighbors above it.

In [ ]:
# much slower than combin(n,k) but gives the same results
def binomial_by_addition(n,k):
    if k > n/2:
        k = n - k
    arr = np.zeros(k+1)
    arr[0] = 1
    for i in range(1,n+1):
        arr[1:] = arr[1:] + arr[:-1]
    return arr[-1]

def binomial_check(n,tol=1E-15):
    match = True
    arr = np.zeros(n+1)
    arr[0] = 1
    for i in range(1,n+1):
        arr[1:i+1] = arr[1:i+1] + arr[:i]
        test = math.factorial(i) / arr[:i+1]
        for m in range(1,i):
            rel_diff = test[m] / (math.factorial(m) * math.factorial(i-m)) - 1
            if abs(rel_diff) > tol:
                match = False
                print('mismatch at combination',i,m,'with relative difference',rel_diff)
    if match: 
        print('No mismatch for combinations of',n,'objects or less')

iters = 1000
for n,k in (12,8),(20,8),(90,30):
    tic = T()
    for iter in range(iters): binomial_by_addition(n,k)
    print(n,k,'\nbinomial_by_addition takes',T()-tic); tic = T()
    for iter in range(iters): math.comb(n,k)
    print('math.comb takes',T()-tic,'\n')
binomial_check(80)

We have now seen that polynomials are relatively easy to work with 
and can usefully approximate a host of non-polynomial functions. 
Numpy offers a class called **Polynomial** that makes operations even easier. 
It works with the coef lists we're familiar with, 
where coef[i] denotes the coefficient on *x***i.
The Polynomial class handles all type conversion.

In [ ]:
from numpy.polynomial import Polynomial as P

coefficients = [-1,0,2]
p = P(coefficients) 
print('The polynomial is',p)         # prints p in Polynomial form
print('Its coefficents are',p.coef)  # prints coefficients as np.array  
print(p)                                    # prints p as an algebraic expression

Evaluation of a Numpy Polynomial is easy. 
So are addition and multiplication of Numpy Polynomials.

In [ ]:
x = np.linspace(1,3,3)
coef1 = [-1,0,2]
p = P(coef1)
print('x =',x,', p(x) =',p(x))
coef2 = [2,0,0,1]
q = P(coef2)
r = p + q
print('q(x) =',q(x),', r(x) = ',r(x))
x = np.linspace(-10,10,200)
max_abs_diff = np.max(np.abs(r(x) - p(x) - q(x)))
print('Maximum absolute difference between r(x) and p(x)+q(x) is',round(max_abs_diff,14))
s = p * q
max_abs_diff = np.max(np.abs(s(x) - p(x) * q(x)))
print('Maximum absolute difference between s(x) and p(x)*q(x) is',round(max_abs_diff,14))

Let's confirm that **Polynomial** calculates (1 + *x*)<sup>*n*</sup> correctly.

In [ ]:
def power_of_xplus1(power):
    xplus1 = P([1,1])
    z = xplus1**power
    binomial = np.ones(power+1)
    for i in range(power):
        binomial[i+1] *= binomial[i] * (power - i) / (i+1)
    allmatch = np.all(binomial == z.coef)
    print('Do all coefficents match the terms in the binomial expansion?',allmatch)
    return z

power = 9
xplus1_power = power_of_xplus1(power)
print(xplus1_power)

Division treats Polynomials like integers, with **//** giving the floor quotient,
**%** giving the remainder, and **divmod** giving both.

In [ ]:
coef1 = [-1,0,2]
p = P(coef1)
coef2 = [2,0,0,1]
q = P(coef2)
quotient = q//p
remainder = q % p
print('Floor quotient q//p = ',quotient,'remainder q%p =',remainder)
quo,rem = divmod(q,p)
print('\nFloor quotient and remainder =',quo,rem)
print('\nDifference between q and quotient*p + remainder =',q - quo * p - rem)

Derivatives and integrals are easily computed using the methods **.deriv** and **.integ**, 
with numbers inserted in the parentheses for repeated operations.

In [ ]:
coef1 = [-1,0,2]
p = P(coef1)
p_der = p.deriv()
p_int = p.integ()
print('p =',p,', derivative =',p_der,', integral =',p_int)
print('Second derivative = ',p_der.deriv(),'or',p.deriv().deriv(),'or',p.deriv(2))
print('Second integral = ',p_int.integ(),'or',p.integ().integ(),'or',p.integ(2))

**Polynomial** can also compute the zeros of a polynomial, 
using the more formal name "roots", or assemble a polynomial from its zeros.

In [ ]:
coef1 = [-1,0,2]
p = P(coef1)
p_zeros = p.roots()
print('Zeros of p estimated at',p_zeros,', where p evaluates to',p(p_zeros))
zeros_at = [1,2,3,5]
q = P.fromroots(zeros_at)
print('Polynomial with zeros at',zeros_at,'is q =',q)
print("Does q evaluate to zero at those points?",all(q(np.array(zeros_at)))==0)

The quotient *r*=*p/q* of two polynomials *p* and *q* 
is known as a rational polynomial, with derivative *r*' = (*p/q*)' = (*p*'*q* - *pq*') / *q*<sup>2</sup>. 
Let's use **Polynomial** to operate with rational polynomials.

In [ ]:
def rat_poly_val(p,q,x):
    return p(x) / q(x)

def deriv_rat_poly(p,q,order=1):
    for n in range(order):
        p = p.deriv() * q - p * q.deriv()
        q = q * q
    return p,q

p = P([0,1])
q = P([1,0,1])
x = np.linspace(-2,2,5)
y = rat_poly_val(p,q,x)
print('x =',x,'y =',y)
deriv = deriv_rat_poly(p,q)
print('\nDerivative of p/q =',deriv[0],'over',deriv[1])

Let's check that this gives the correct answer for the derivative of *x*<sup>-2</sup> = 1 / *x*<sup>2</sup>.

In [ ]:
p = P(1)
q = P([0,0,1])
print('True derivative of x**-2 is -2x**-3, or',P(-2),'over',P([0,0,0,1]))
deriv = deriv_rat_poly(p,q)
print('\nOur algorithm says it equals',deriv[0],'over',deriv[1])

The previous answer is correct but has an extra factor of *x* in the numerator and denominator.
This code factors out common terms in *x - root*.

In [ ]:
def simplify(p,q,root):
    while p(root) == q(root) == 0:
        p = p // P([-root,1])
        q = q // P([-root,1])
    return p,q

p = P(1)
q = P([0,0,1])
print('True derivative of x**-2 is -2x**-3, or',P(-2),'over',P([0,0,0,1]))
deriv = deriv_rat_poly(p,q)
deriv = simplify(deriv[0],deriv[1],0)
print('\nOur revised algorithm says it equals',deriv[0],'over',deriv[1])

Let's now check the derivatives of *x*<sup>*-n*</sup> for higher whole numbers *n*.

In [ ]:
p = P(1)
q = P([0,0,1])
for n in range(3,11):
    q = q * P([0,1])
    deriv = deriv_rat_poly(p,q)
    deriv = simplify(deriv[0],deriv[1],0)
    print('The derivative of x**',-n,'equals',deriv[0],'over',deriv[1])

Let's use these algorithms to compute Taylor series for rational polynomials.

In [ ]:
def taylor_rat_poly(p,q,a,order):
    coef = [rat_poly_val(p,q,a)]
    for n in range(1,order+1):
        p,q = deriv_rat_poly(p,q)
        p,q = simplify(p,q,a)
        deriv_val = rat_poly_val(p,q,a)
        coef.append(deriv_val/math.factorial(n))
    return P(coef)

p = P([1])
q = P([0,0,1])
r = taylor_rat_poly(p,q,1,10)
print(r)